<a href="https://colab.research.google.com/github/Wanrapee6730406291/agentic-rag-workshop/blob/main/agentic_rag_4hr_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megacare-dev/agentic_rag_workshop/blob/main/th/4hr/agentic_rag_4hr_homework.ipynb)

# 📝 แบบฝึกหัด: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า embedding, score, และ agent response ที่ต้องตรงกับรหัสนักศึกษา

## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'scikit-learn']:
    _pip_install(_pkg)

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

  ⏭️  google-adk: skipped
  ⏭️  google-genai: skipped
  ⏭️  sentence-transformers: skipped
  ✅ qdrant-client: done
  ✅ langchain-text-splitters: done
  ⏭️  scikit-learn: skipped
✅ ติดตั้งเรียบร้อย!
CPU times: user 1.25 s, sys: 190 ms, total: 1.44 s
Wall time: 19.1 s


## 🎓 กรอกข้อมูลนักศึกษา

In [ ]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = 'วันรพี ไชยคำ'   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = '6730406291'   # เช่น '6512345678'
PHONE        = '095-376-1836'   # เช่น '081-234-5678'
LINE_ID      = 'nam22wc'   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

✅ ชื่อ-นามสกุล: วันรพี ไชยคำ
✅ รหัสนักศึกษา: 6730406291
📱 เบอร์โทร: 095-376-1836
💬 LINE ID: nam22wc


## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [ ]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

✅ สร้างข้อมูลเฉพาะสำหรับ 6730406291
📁 จำนวนไฟล์: 8 ไฟล์
🔍 Query เฉพาะตัว: "วิธีการแบ่งเอกสารเป็นส่วนย่อย"
  📄 doc_1.txt (149 ตัวอักษร)
  📄 doc_2.txt (150 ตัวอักษร)
  📄 doc_3.txt (141 ตัวอักษร)
  📄 doc_4.txt (152 ตัวอักษร)
  📄 doc_5.txt (136 ตัวอักษร)
  📄 doc_6.txt (164 ตัวอักษร)
  📄 doc_7.txt (146 ตัวอักษร)
  📄 doc_8.txt (146 ตัวอักษร)
CPU times: user 860 µs, sys: 2 ms, total: 2.86 ms
Wall time: 4.34 ms


---
## 🎯 ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน)

- รวมข้อความจากทุกไฟล์ใน `homework_data/`
- Chunk ด้วย `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- สร้าง Embedding ด้วย `intfloat/multilingual-e5-large`
- ค้นหาด้วย query: `MY_QUERY` (ที่สร้างจากรหัสนักศึกษา)
- เก็บลง Qdrant collection ชื่อ `f'hw_{STUDENT_ID}'`

**📝 รายงาน:**
1. ได้ทั้งหมดกี่ chunks?
2. Chunk ไหนมี similarity สูงสุด? (score ทศนิยม 4 ตำแหน่ง)
3. Top-3 ผลลัพธ์จาก Qdrant มี score เท่าไร?

In [ ]:
import glob
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from qdrant_client import QdrantClient, models

#อ่านไฟล์ homework_data/
file_paths = sorted(glob.glob('homework_data/*.txt'))
all_paragraphs_text = []
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        all_paragraphs_text.append(f.read().strip())
all_text = "\n" .join(all_paragraphs_text)

#chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_text(all_text)

#embedding
model = SentenceTransformer('intfloat/multilingual-e5-large')
passages = ['passage: '+ c for c in chunks]
chunk_embeddings = model.encode(passages, normalize_embeddings=True)

query_text = f'query: {MY_QUERY}'
query_embedding = model.encode(query_text, normalize_embeddings=True)

#คำนวณ หา chunk ไหนมี similarity สูงสุด
sim_scores = cosine_similarity([query_embedding], chunk_embeddings)[0]
best_chunk_idx = int(np.argmax(sim_scores))
max_similarity_score = sim_scores[best_chunk_idx]

#เก็บลง Qdrant collection ชื่อ f'hw_{STUDENT_ID}'
collection_name= f'hw_{STUDENT_ID}'
qdrant_client = QdrantClient(":memory:")
vector_size = chunk_embeddings.shape[1]

#สร้าง collection ใหม่
qdrant_client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE
    )
)

#Update chunks เข้า Qdrant
points =[
    models.PointStruct(
        id=i,
        vector=chunk_embeddings[i].tolist(),
        payload={"text": chunks[i]}
    )
    for i in range(len(chunks))

]
qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

#ค้นหา Top-3 ผลลัพธ์จาก Qdrant
qdrant_results = qdrant_client.query_points(
    collection_name=collection_name,
    query = query_embedding.tolist(),
    limit=3
).points

#รายงานผล
print("="*60)
print("รายงานผลขั้นตอนที่1:")
print(f"1. จำนวน chunks: {len(chunks)}chunks")
print(f"2. Chunk ที่มี similarity สูงสุด (Cosine Similarity):")
print(f"   - Index: {best_chunk_idx}")
print(f"   - Score: {max_similarity_score:.4f}")
print(f"\n3. Top-3 ผลลัพธ์จาก Qdrant:")
for rank, res in enumerate(qdrant_results, start=1):
    print(f"   อันดับ {rank}: Score = {res.score:.4f} | Text = {res.payload['text']}")
print("="*60)

#Self-check
assert len(chunks)>0, 'ยังไม่ได้ chunk'
assert len(qdrant_results)==3, 'ควรได้ top_k=3 จาก Qdrant'
print(f'Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

รายงานผลขั้นตอนที่1:
1. จำนวน chunks: 11chunks
2. Chunk ที่มี similarity สูงสุด (Cosine Similarity):
   - Index: 6
   - Score: 0.8446

3. Top-3 ผลลัพธ์จาก Qdrant:
   อันดับ 1: Score = 0.8446 | Text = Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic
   อันดับ 2: Score = 0.7955 | Text = NLP และ Information Retrieval
   อันดับ 3: Score = 0.7818 | Text = LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้
Step 1 passed: 11 chunks, top score=0.8446


/tmp/ipykernel_2219/3249531195.py:39: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


---
## 🎯 ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน)

- ตั้งค่า Gemini API Key (Colab Secrets)
- สร้าง **Custom Tool** อย่างน้อย 1 ตัว (ห้ามซ้ำกับ BMI ในคาบ)
- สร้าง **Agent** ด้วย Google ADK ที่ใช้ Tool ได้
- ทดลองคุย → แสดงว่า Agent เรียก Tool ได้จริง

**📝 รายงาน:**
1. Tool ของคุณทำอะไร? (อธิบาย 1-2 ประโยค)
2. แสดง output ที่ Agent เรียก Tool สำเร็จ
3. ทำไม docstring ถึงสำคัญ? (อธิบาย 1-2 ประโยค)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 วาง API Key: ')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# ─── สร้าง Custom Tool (คำนวณงบประมาณไปคอนเสิร์ต) ───
def concert_budget_planner(ticket_price: float, travel_cost: float, food_cost: float, hotel_cost: float = 0.0, shopping_cost: float = 0.0, budget: float = 2500.0) -> str:
    """คำนวณและตรวจสอบค่าใช้จ่ายสำหรับการไปคอนเสิร์ตเปรียบเทียบกับงบประมาณที่มี

    Args:
        ticket_price: ค่าบัตรคอนเสิร์ต (บาท) เช่น 500
        travel_cost: ค่าเดินทางไป-กลับ (บาท) เช่น 900
        food_cost: ค่าอาหาร เครื่องดื่ม (บาท) เช่น 400
        hotel_cost: ค่าที่พัก (บาท) เช่น 500 (ถ้าไม่มีให้ใส่ 0)
        shopping_cost: ค่าช้อปปิ้งของแท่งไฟ/ของที่ระลึก (บาท) เช่น 200 (ถ้าไม่มีให้ใส่ 0)
        budget: งบประมาณรวมที่มี (บาท) ค่าเริ่มต้นคือ 2500 บาท
    """
    total_expense = ticket_price + travel_cost + food_cost + hotel_cost + shopping_cost
    balance = budget - total_expense

    if balance >= 0:
        status = f"✅ อยู่ในงบ! เหลือเงินเก็บ/สำรอง {balance:,.2f} บาท"
    else:
        status = f"❌ งบบานปลาย! เกินงบไป {abs(balance):,.2f} บาท (ควรปรับลดค่าใช้จ่าย)"

    return (
        f"งบประมาณตั้งต้น: {budget:,.2f} บาท | "
        f"ค่าบัตร: {ticket_price:,.2f} บาท | "
        f"ค่าเดินทาง: {travel_cost:,.2f} บาท | "
        f"ค่าอาหาร: {food_cost:,.2f} บาท | "
        f"ค่าที่พัก: {hotel_cost:,.2f} บาท | "
        f"ค่าช้อปปิ้ง: {shopping_cost:,.2f} บาท | "
        f"รวมค่าใช้จ่าย: {total_expense:,.2f} บาท | "
        f"ผลลัพธ์: {status}"
    )

tool = FunctionTool(concert_budget_planner)

# ─── สร้าง Agent ───
my_agent = LlmAgent(
    name='concert_planner_assistant',
    model='gemini-3.1-flash-lite', # เปลี่ยนมาใช้ gemini-3.1-flash-lite เพื่อป้องกัน Error 404 และปัญหาโควต้า
    instruction='คุณเป็นผู้ช่วยวางแผนการเงินไปคอนเสิร์ต หากผู้ใช้ถามเรื่องการวางแผนค่าใช้จ่ายไปคอนเสิร์ต จัดสรรงบ 2500 บาท ให้เรียกใช้เครื่องมือ concert_budget_planner เสมอ',
    tools=[tool]
)

# ─── ทดสอบ ───
async def chat_with_agent(agent, message):
    runner = InMemoryRunner(agent=agent, app_name='homework')
    session = await runner.session_service.create_session(
        app_name='homework', user_id='student'
    )
    content = genai_types.Content(
        role='user', parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# ถามคำถามที่ต้องใช้ tool (ทดสอบงบ 2,500 บาท)
prompt_message = 'มีงบไปคอนเสิร์ต 2,500 บาท ซื้อบัตรไป 500 บาท ค่าเดินทาง 900 บาท ซื้อกินอีก 400 บาท พอไหม เหลือหรือเกินเท่าไหร่?'
answer = await chat_with_agent(my_agent, prompt_message)

print("=" * 60)
print(f"❓ คำถาม: {prompt_message}")
print(f"🤖 Agent Response:\n{answer}")
print("=" * 60)

# 📝 รายงานสรุป
print("\n📝 รายงานผลขั้นตอนที่ 2:")
print("1. Tool ของคุณทำอะไร?:")
print("   - เป็นเครื่องมือวางแผนและคำนวณงบประมาณไปคอนเสิร์ต (Concert Budget Planner) โดยรับค่าใช้จ่ายรายการต่างๆ (บัตร, เดินทาง, กิน, ที่พัก, ช้อปปิ้ง) มาคำนวณเปรียบเทียบกับงบประมาณ 2,500 บาทว่าพอหรือไม่")
print("\n2. ทำไม docstring ถึงสำคัญ?:")
print("   - เพราะ LLM ใช้ docstring เป็นคำอธิบาย (Function Description) ในการตัดสินใจว่า Tool นี้ใช้จัดการงบอย่างไร รวมถึงระบุประเภท Parameter แต่ละตัวเพื่อดึงตัวเลขค่าใช้จ่ายจากข้อความผู้ใช้มาส่งให้ฟังก์ชันคำนวณได้อย่างถูกต้อง")

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent ไม่ตอบ'
print(f'\n✅ Step 2 passed!')

❓ คำถาม: มีงบไปคอนเสิร์ต 2,500 บาท ซื้อบัตรไป 500 บาท ค่าเดินทาง 900 บาท ซื้อกินอีก 400 บาท พอไหม เหลือหรือเกินเท่าไหร่?
🤖 Agent Response:
สำหรับแผนการไปคอนเสิร์ตของคุณ สรุปได้ดังนี้ครับ:

คุณมีค่าใช้จ่ายรวมอยู่ที่ **1,800 บาท** จากงบประมาณทั้งหมด 2,500 บาท

**สรุปสถานะ:**
✅ **งบประมาณเพียงพอครับ!** คุณจะเหลือเงินเก็บหรือไว้เป็นค่าใช้จ่ายสำรองอีก **700 บาท** ครับ

📝 รายงานผลขั้นตอนที่ 2:
1. Tool ของคุณทำอะไร?:
   - เป็นเครื่องมือวางแผนและคำนวณงบประมาณไปคอนเสิร์ต (Concert Budget Planner) โดยรับค่าใช้จ่ายรายการต่างๆ (บัตร, เดินทาง, กิน, ที่พัก, ช้อปปิ้ง) มาคำนวณเปรียบเทียบกับงบประมาณ 2,500 บาทว่าพอหรือไม่

2. ทำไม docstring ถึงสำคัญ?:
   - เพราะ LLM ใช้ docstring เป็นคำอธิบาย (Function Description) ในการตัดสินใจว่า Tool นี้ใช้จัดการงบอย่างไร รวมถึงระบุประเภท Parameter แต่ละตัวเพื่อดึงตัวเลขค่าใช้จ่ายจากข้อความผู้ใช้มาส่งให้ฟังก์ชันคำนวณได้อย่างถูกต้อง

✅ Step 2 passed!


---
## 🎯 ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน)

- สร้าง **RAG Tool** ที่ค้นจาก Qdrant (ใช้ collection จากขั้นตอนที่ 1)
- สร้าง **RAG Agent** ที่ใช้ RAG Tool ตอบคำถาม
- ถามคำถาม 3 ข้อ (กำหนดให้) → บันทึกคำตอบ
- ให้คะแนนคำตอบด้วย **LLM-as-Judge** (ใช้ Gemini ให้คะแนน 1-5)

**คำถามที่ต้องถาม:**
```python
questions = [
    f'query: {MY_QUERY}',   # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]
```

**📝 รายงาน:**
1. คำตอบของ RAG Agent ต่อ 3 คำถาม
2. LLM-as-Judge ให้คะแนนเท่าไร? (1-5 ต่อข้อ)
3. อธิบาย: Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร? (2-3 ประโยค)

In [ ]:
# ขั้นตอนที่ 3: เติมโค้ดในที่ว่าง

import json
from google import genai
from google.genai import types as genai_types
from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool

# ─── A) สร้าง RAG Tool ───
def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ที่จัดเก็บใน Qdrant
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI, Machine Learning, NLP

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    # 1. แปลง query เป็น embedding โดยใช้ prefix 'query: ' ตามมาตรฐาน E5 Model
    query_emb = model.encode(f'query: {query}', normalize_embeddings=True)

    # 2. ค้นหา Top-3 ผลลัพธ์จาก Qdrant Collection ที่สร้างไว้ใน Step 1
    collection_name = f'hw_{STUDENT_ID}'
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_emb.tolist(),
        limit=3
    ).points

    # 3. จัดรูปแบบผลลัพธ์เป็น String ส่งกลับให้ Agent
    res_str = []
    for rank, point in enumerate(results, start=1):
        res_str.append(f"[{rank}] Score: {point.score:.4f} | Content: {point.payload['text']}")

    return "\n".join(res_str)

# ─── B) สร้าง RAG Agent ───
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-3.1-flash-lite',  # เปลี่ยนมาใช้ gemini-3.1-flash-lite
    instruction='คุณเป็น AI ที่ตอบคำถามจากฐานความรู้ ใช้ tool search_knowledge ค้นหาข้อมูลก่อนตอบเสมอ ตอบเป็นภาษาไทย',
    tools=[FunctionTool(search_knowledge)]
)

# ─── C) ถาม 3 คำถาม ───
questions = [
    MY_QUERY,  # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]

rag_answers = []
for q in questions:
    ans = await chat_with_agent(rag_agent, q)
    rag_answers.append({'question': q, 'answer': ans})
    print(f'\n{"="*50}')
    print(f'❓ {q}')
    print(f'🤖 {ans}')

# ─── D) LLM-as-Judge ───
from google import genai
judge_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

# 📋 Template สำหรับ judge (ใช้ตามนี้ได้เลย)
JUDGE_PROMPT = '''คุณเป็นผู้ตรวจคุณภาพคำตอบ AI

คำถาม: {question}
คำตอบ: {answer}

ให้คะแนน 1-5 ตามเกณฑ์:
- 5 = ถูกต้อง ครบถ้วน อธิบายชัดเจน
- 4 = ถูกต้อง แต่ขาดรายละเอียดบางส่วน
- 3 = ถูกบางส่วน มีข้อผิดพลาดเล็กน้อย
- 2 = ตอบไม่ตรงประเด็น หรือผิดหลายจุด
- 1 = ผิดทั้งหมด หรือไม่ตอบ

ตอบ JSON: {{"score": 0, "reason": "..."}}
'''

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
for qa in rag_answers:
    prompt = JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer'])
    resp = judge_client.models.generate_content(
        model='gemini-3.1-flash-lite', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
    )
    result = json.loads(resp.text)
    print(f'  ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')

# 📝 รายงานสรุปเพิ่มเติม
print(f'\n{"="*50}')
print("📝 รายงานสรุปขั้นตอนที่ 3:")
print("1. คำตอบของ RAG Agent ต่อ 3 คำถาม: บันทึกเรียบร้อยใน rag_answers")
print("2. LLM-as-Judge คะแนน: ได้รับคะแนนประเมินช่วง 4-5/5 ในแต่ละข้อตามความสมบูรณ์ของบริบท")
print("3. อธิบาย: Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร?:")
print("   Agent จะวิเคราะห์เจตนาของคำถามผู้ใช้เปรียบเทียบกับ Docstring ของเครื่องมือ search_knowledge เมื่อพบว่าผู้ใช้ต้องการข้อมูลในบทเรียน Agent จะตัดสินใจทำ Tool Calling ส่งคำถามไปแปลงเป็น Embedding แล้วทำ Cosine Similarity Search ดึง Top-3 Chunks จาก Qdrant มาประมวลผลเป็นคำตอบ")

# ✅ Self-check
assert len(rag_answers) == 3, '❌ ต้องตอบ 3 คำถาม'
print(f'\n✅ Step 3 passed: ตอบครบ 3 ข้อ + LLM-as-Judge เสร็จ!')


❓ วิธีการแบ่งเอกสารเป็นส่วนย่อย
🤖 การแบ่งเอกสารเป็นส่วนย่อย (Chunking) เป็นขั้นตอนสำคัญในการทำ RAG (Retrieval-Augmented Generation) เพื่อให้ระบบสามารถดึงข้อมูลที่เกี่ยวข้องไปใช้กับ LLM ได้อย่างมีประสิทธิภาพ โดยวิธีหลักๆ ที่นิยมใช้มีดังนี้ครับ:

1.  **Fixed-size Chunking (การแบ่งตามขนาดคงที่):**
    *   **วิธีการ:** แบ่งข้อความตามจำนวนตัวอักษรหรือจำนวน Token ที่กำหนดไว้ตายตัว (เช่น ทุกๆ 500 หรือ 1000 tokens)
    *   **ข้อดี:** ทำได้ง่าย รวดเร็ว และควบคุมขนาดของ Chunk ได้แน่นอน
    *   **ข้อควรระวัง:** อาจทำให้ประโยคหรือใจความสำคัญถูกตัดขาดกลางคัน จึงนิยมทำ **Overlap** (ให้ส่วนท้ายของ Chunk แรก ซ้อนกับส่วนต้นของ Chunk ถัดไป) เพื่อรักษาบริบทของข้อมูล

2.  **Recursive Chunking (การแบ่งแบบเรียกซ้ำ):**
    *   **วิธีการ:** เป็นการแบ่งที่ซับซ้อนขึ้นโดยพยายามรักษาโครงสร้างของเอกสาร โดยจะพยายามแบ่งตามตัวคั่น (Separators) จากลำดับความสำคัญ เช่น ย่อหน้า (Paragraph), ประโยค (Sentence), และเว้นวรรค (Space) จนกว่าจะได้ขนาด Chunk ที่เหมาะสม
    *   **ข้อดี:** ช่วยให้เนื้อหาในแต่ละ Chunk มีความเป็นธร

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงานได้, ผล Qdrant ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent ตอบครบ 3 ข้อ, LLM-as-Judge ให้คะแนน, อธิบาย |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [✅] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [✅] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน')
print('  [✅] ขั้นตอนที่ 2: Agent + Tool ทำงาน')
print('  [✅] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge')
print('  [✅] ทุก cell run แล้วมีผลลัพธ์')

👤 ชื่อ-นามสกุล: วันรพี ไชยคำ
🎓 รหัสนักศึกษา: 6730406291
📱 เบอร์โทร: 095-376-1836
💬 LINE ID: nam22wc
🔑 Verification Code: d26765c157e9
📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.

📋 Checklist ก่อนส่ง:
  [✅] กรอกข้อมูลส่วนตัวครบถ้วน
  [✅] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน
  [✅] ขั้นตอนที่ 2: Agent + Tool ทำงาน
  [✅] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge
  [✅] ทุก cell run แล้วมีผลลัพธ์
